In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold


# ================================================================
# Configuration
# ================================================================

PROJECT_DIR = Path(r"C:\New folder\New Emodect")

CLEANED_DIR = PROJECT_DIR / "cleaned_metadata"
EDA_DIR = PROJECT_DIR / "reports" / "eda"
SPLIT_DIR = CLEANED_DIR / "splits"

EMOTION_MANIFEST = (
    CLEANED_DIR /
    "final_emotion_training_manifest.csv"
)

SARCASM_MANIFEST = (
    CLEANED_DIR /
    "sarcasm" /
    "final_sarcasm_training_manifest.csv"
)

os.makedirs(
    EDA_DIR,
    exist_ok=True
)

os.makedirs(
    SPLIT_DIR,
    exist_ok=True
)



# ================================================================
# Current pipeline state (September 2026)
# ================================================================
#
# The upstream cleaned manifests were rebuilt after:
#   - FER2013 parser correction
#   - GoEmotions restoration as text
#   - current seven-class canonical mapping
#   - current independent sarcasm cleaning
#
# Current authoritative manifest sizes:
#   Emotion  : 146,335 rows
#   Sarcasm  : 29,183 rows
#
# These are validation guards only. The split notebook never
# modifies the upstream manifests.
# ================================================================

EXPECTED_EMOTION_ROWS = 146_335
EXPECTED_SARCASM_ROWS = 29_183


# ================================================================
# Authoritative final split targets
# ================================================================
#
# These exact targets are required by the downstream preprocessing
# contract. The split allocator first performs deterministic
# StratifiedGroupKFold assignment, then performs a minimal,
# deterministic size-repair using only singleton split groups.
# No multi-row group is ever broken.
#
EXPECTED_EMOTION_SPLITS = {
    "train": 99_961,
    "validation": 24_679,
    "test": 21_695,
}

EXPECTED_SARCASM_SPLITS = {
    "train": 20_254,
    "validation": 4_479,
    "test": 4_450,
}

REQUIRED_EMOTION_DATASETS = {
    "CREMA-D",
    "RAVDESS",
    "SAVEE",
    "TESS",
    "IEMOCAP",
    "FER2013",
    "AffectNet",
    "CK+",
    "RAF-DB",
    "GoEmotions",
}

# GoEmotions was independently audited after the one-hot parser
# correction. The corrected cleaner produced 133,688 canonical
# rows before global deduplication; 50,831 GoEmotions rows remain
# in the frozen 146,335-row final emotion manifest.
EXPECTED_GOEMOTIONS_FINAL_ROWS = 50_831

# ================================================================
# Canonical project targets
# ================================================================

EMOTION_CLASSES = [
    "anger",
    "disgust",
    "fear",
    "happiness",
    "sadness",
    "surprise",
    "neutral",
]

SARCASM_CLASSES = [
    "non_sarcastic",
    "sarcastic",
]


# ================================================================
# Architecture rule
# ================================================================
#
# Emotion:
#   Seven-class primary emotion target.
#
# Sarcasm:
#   Independent binary capability.
#   It is NEVER merged into the emotion target.
#
# Therefore:
#
#   Emotion model  -> 7 classes
#   Sarcasm model  -> 2 classes / detection flag
#
# ================================================================

In [2]:
from pathlib import Path
import pandas as pd

sarcasm_path = Path(
    r"C:\New folder\New Emodect\cleaned_metadata\sarcasm\final_sarcasm_training_manifest.csv"
)

df = pd.read_csv(sarcasm_path)

print("File:", sarcasm_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicates:", df.duplicated().sum())
print("\nDataset counts:")
print(df["dataset"].value_counts(dropna=False))
print("\nSarcasm label counts:")
print(df["sarcasm"].value_counts(dropna=False))

File: C:\New folder\New Emodect\cleaned_metadata\sarcasm\final_sarcasm_training_manifest.csv
Rows: 29183
Columns: 15
Duplicates: 0

Dataset counts:
dataset
NewsHeadlinesSarcasm    28503
MUStARD                   680
Name: count, dtype: int64

Sarcasm label counts:
sarcasm
0    15288
1    13895
Name: count, dtype: int64


In [3]:
# ================================================================
# Metadata loading and leakage-safe group construction
# ================================================================

def normalize_path(value):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return ""
    return os.path.normcase(
        os.path.normpath(str(value))
    )


def find_column(df, candidates):
    """
    Return the first matching column, case-insensitively.
    """
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in candidates:
        key = str(name).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


# ================================================================
# Emotion manifest loader
# ================================================================

EMOTION_LABEL_CANDIDATES = [
    "mapped_emotion",
    "emotion",
    "emotion_label",
    "label",
    "target",
]


def load_emotion_manifest(path):

    df = pd.read_csv(path)

    # The cleaned metadata manifest uses `mapped_emotion`
    # as the canonical seven-class emotion label.
    #
    # Internally normalize this to `emotion` so that all
    # downstream split code has one consistent target name.

    col = find_column(
        df,
        EMOTION_LABEL_CANDIDATES
    )

    if col is None:
        raise ValueError(
            "Emotion manifest has no recognizable label column.\n"
            f"Expected one of: {EMOTION_LABEL_CANDIDATES}\n"
            f"Available columns: {list(df.columns)}"
        )

    source_column = col

    if col != "emotion":
        df = df.rename(
            columns={col: "emotion"}
        )

    # Normalize labels
    df["emotion"] = (
        df["emotion"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Validate against the seven canonical project classes
    bad = sorted(
        set(df["emotion"].dropna())
        - set(EMOTION_CLASSES)
    )

    if bad:
        raise ValueError(
            "Unexpected emotion labels found:\n"
            f"{bad}\n\n"
            f"Expected classes:\n"
            f"{EMOTION_CLASSES}"
        )

    # Store provenance without altering the source file
    df.attrs["source_label_column"] = source_column

    return df.reset_index(drop=True)


# ================================================================
# Sarcasm manifest loader
# ================================================================

SARCASM_LABEL_CANDIDATES = [
    "mapped_sarcasm",
    "sarcasm_label",
    "sarcasm",
    "label",
    "target",
]


def normalize_sarcasm_label(value):
    """
    Normalize all supported sarcasm label representations
    into the project's binary labels:

        0 -> non_sarcastic
        1 -> sarcastic

    Also accepts common textual representations.
    """

    if pd.isna(value):
        return None

    value_str = str(value).strip().lower()

    # ------------------------------------------------------------
    # Numeric representation
    # ------------------------------------------------------------

    if value_str in {
        "0",
        "0.0",
        "false",
        "non_sarcastic",
        "non-sarcastic",
        "nonsarcastic",
        "not_sarcastic",
        "not-sarcastic",
        "no",
    }:
        return "non_sarcastic"

    if value_str in {
        "1",
        "1.0",
        "true",
        "sarcastic",
        "sarcasm",
        "yes",
    }:
        return "sarcastic"

    return value_str


def load_sarcasm_manifest(path):

    df = pd.read_csv(path)

    col = find_column(
        df,
        SARCASM_LABEL_CANDIDATES
    )

    if col is None:
        raise ValueError(
            "Sarcasm manifest has no recognizable label column.\n"
            f"Expected one of: {SARCASM_LABEL_CANDIDATES}\n"
            f"Available columns: {list(df.columns)}"
        )

    source_column = col

    if col != "sarcasm_label":
        df = df.rename(
            columns={col: "sarcasm_label"}
        )

    # Normalize numeric/text labels to the canonical
    # binary sarcasm representation.
    df["sarcasm_label"] = (
        df["sarcasm_label"]
        .apply(normalize_sarcasm_label)
    )

    # Detect invalid values after normalization
    bad = sorted(
        set(df["sarcasm_label"].dropna())
        - set(SARCASM_CLASSES)
    )

    if bad:
        raise ValueError(
            "Unexpected sarcasm labels found after normalization:\n"
            f"{bad}\n\n"
            "Supported representations include:\n"
            "0 / 1\n"
            "false / true\n"
            "non_sarcastic / sarcastic"
        )

    # Hard architectural separation:
    # sarcasm is an independent task and must not become
    # one of the seven emotion targets.
    if "emotion" in df.columns:
        raise ValueError(
            "Sarcasm manifest must NOT contain an emotion target."
        )

    if "mapped_emotion" in df.columns:
        raise ValueError(
            "Sarcasm manifest must NOT contain a mapped_emotion target."
        )

    df.attrs["source_label_column"] = source_column

    return df.reset_index(drop=True)


# ================================================================
# Emotion leakage-safe group construction
# ================================================================

def derive_emotion_group(row, idx):
    """
    Determine the group identity used for train/validation/test
    splitting.

    Priority:

        1. Existing split_group_id
        2. Existing reliable group_id
        3. Dataset-specific fallback
        4. Sample-only fallback

    The existing split_group_id is preferred because it was
    generated during the cleaned metadata stage.
    """

    dataset = str(
        row.get("dataset", "")
    ).strip()

    # ------------------------------------------------------------
    # 1. Prefer the frozen split_group_id
    # ------------------------------------------------------------

    existing_split = row.get(
        "split_group_id",
        None
    )

    if (
        pd.notna(existing_split)
        and str(existing_split).strip()
        and str(existing_split).strip().lower()
        not in {"nan", "none"}
    ):
        return str(
            existing_split
        ).strip()

    # ------------------------------------------------------------
    # 2. Fall back to group_id
    # ------------------------------------------------------------

    existing_group = row.get(
        "group_id",
        None
    )

    if (
        pd.notna(existing_group)
        and str(existing_group).strip()
        and str(existing_group).strip().lower()
        not in {"nan", "none"}
        and not str(existing_group).strip().startswith(
            "sample_only"
        )
    ):
        return str(
            existing_group
        ).strip()

    # ------------------------------------------------------------
    # 3. Dataset-specific fallback
    # ------------------------------------------------------------

    fp = normalize_path(
        row.get("file_path", None)
    )

    stem = (
        Path(fp).stem
        if fp
        else ""
    )

    # ------------------------------------------------------------
    # CREMA-D
    # ------------------------------------------------------------

    if dataset == "CREMA-D":

        parts = stem.split("_")

        if parts and parts[0]:
            return (
                f"CREMA-D_{parts[0]}"
            )

        return (
            f"CREMA-D_sample_{idx}"
        )

    # ------------------------------------------------------------
    # RAVDESS
    # ------------------------------------------------------------

    if dataset == "RAVDESS":

        parts = stem.split("-")

        if len(parts) >= 7:
            return (
                f"RAVDESS_Actor_{parts[6]}"
            )

        return (
            f"RAVDESS_sample_{idx}"
        )

    # ------------------------------------------------------------
    # SAVEE
    # ------------------------------------------------------------

    if dataset == "SAVEE":

        if len(stem) >= 2:
            return (
                f"SAVEE_{stem[:2]}"
            )

        return (
            f"SAVEE_sample_{idx}"
        )

    # ------------------------------------------------------------
    # TESS
    # ------------------------------------------------------------

    if dataset == "TESS":

        parent = (
            Path(fp).parent.name
            if fp
            else ""
        )

        if parent:

            speaker = (
                parent.split("_")[0]
            )

            if speaker:
                return (
                    f"TESS_{speaker}"
                )

        return (
            f"TESS_sample_{idx}"
        )

    # ------------------------------------------------------------
    # IEMOCAP
    # ------------------------------------------------------------

    if dataset == "IEMOCAP":

        match = re.search(
            r"Session(\d+)",
            fp,
            flags=re.IGNORECASE
        )

        if match:
            return (
                f"IEMOCAP_Session{match.group(1)}"
            )

        return (
            f"IEMOCAP_sample_{idx}"
        )

    # ------------------------------------------------------------
    # Datasets without defensible subject identity
    # ------------------------------------------------------------

    return (
        f"sample_only_{dataset}_{idx}"
    )


# ================================================================
# Sarcasm leakage-safe group construction
# ================================================================

def derive_sarcasm_group(row, idx):
    """
    Construct leakage-safe groups for the independent sarcasm
    detection task.
    """

    existing = row.get(
        "group_id",
        None
    )

    if (
        pd.notna(existing)
        and str(existing).strip()
        and str(existing).strip().lower()
        not in {"nan", "none"}
    ):
        return str(
            existing
        ).strip()

    dataset = str(
        row.get("dataset", "")
    ).strip()

    # ------------------------------------------------------------
    # MUStARD
    # ------------------------------------------------------------

    if dataset == "MUStARD":

        speaker = str(
            row.get(
                "speaker",
                "unknown"
            )
        ).strip()

        show = str(
            row.get(
                "show",
                "unknown"
            )
        ).strip()

        return (
            f"MUStARD_{show}_{speaker}"
        )

    # ------------------------------------------------------------
    # News Headlines
    # ------------------------------------------------------------

    return (
        f"sample_only_{dataset}_{idx}"
    )


# ================================================================
# Complete manifest loading
# ================================================================

def load_manifests():

    # ------------------------------------------------------------
    # Verify manifest files
    # ------------------------------------------------------------

    if not EMOTION_MANIFEST.exists():
        raise FileNotFoundError(
            f"Missing emotion manifest:\n"
            f"{EMOTION_MANIFEST}"
        )

    if not SARCASM_MANIFEST.exists():
        raise FileNotFoundError(
            f"Missing sarcasm manifest:\n"
            f"{SARCASM_MANIFEST}"
        )

    # ------------------------------------------------------------
    # Load manifests
    # ------------------------------------------------------------

    emotion = load_emotion_manifest(
        EMOTION_MANIFEST
    )

    sarcasm = load_sarcasm_manifest(
        SARCASM_MANIFEST
    )

    # ------------------------------------------------------------
    # Construct leakage-safe groups
    # ------------------------------------------------------------

    emotion["split_group"] = [
        derive_emotion_group(
            row,
            idx
        )
        for idx, (_, row)
        in enumerate(
            emotion.iterrows()
        )
    ]

    sarcasm["split_group"] = [
        derive_sarcasm_group(
            row,
            idx
        )
        for idx, (_, row)
        in enumerate(
            sarcasm.iterrows()
        )
    ]

    # ------------------------------------------------------------
    # Read manifest headers once
    # ------------------------------------------------------------

    emotion_header = pd.read_csv(
        EMOTION_MANIFEST,
        nrows=0
    )

    sarcasm_header = pd.read_csv(
        SARCASM_MANIFEST,
        nrows=0
    )

    emotion_source_label = find_column(
        emotion_header,
        EMOTION_LABEL_CANDIDATES
    )

    sarcasm_source_label = find_column(
        sarcasm_header,
        SARCASM_LABEL_CANDIDATES
    )

    # ------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------

    print(
        f"Emotion manifest rows: "
        f"{len(emotion):,}"
    )

    print(
        f"Sarcasm manifest rows: "
        f"{len(sarcasm):,}"
    )

    print(
        "Emotion label column normalized from: "
        f"{emotion_source_label}"
    )

    print(
        "Sarcasm label column normalized from: "
        f"{sarcasm_source_label}"
    )

    print(
        "Emotion unique split groups: "
        f"{emotion['split_group'].nunique():,}"
    )

    print(
        "Sarcasm unique split groups: "
        f"{sarcasm['split_group'].nunique():,}"
    )

    # ------------------------------------------------------------
    # Emotion integrity
    # ------------------------------------------------------------

    assert (
        emotion["emotion"]
        .isin(EMOTION_CLASSES)
        .all()
    ), (
        "Invalid emotion labels detected."
    )

    # ------------------------------------------------------------
    # Sarcasm integrity
    # ------------------------------------------------------------

    assert (
        sarcasm["sarcasm_label"]
        .isin(SARCASM_CLASSES)
        .all()
    ), (
        "Invalid sarcasm labels detected."
    )

    # ------------------------------------------------------------
    # Group integrity
    # ------------------------------------------------------------

    assert (
        emotion["split_group"]
        .notna()
        .all()
    ), (
        "Missing emotion split groups detected."
    )

    assert (
        sarcasm["split_group"]
        .notna()
        .all()
    ), (
        "Missing sarcasm split groups detected."
    )

    # ------------------------------------------------------------
    # Current manifest size guards
    # ------------------------------------------------------------

    if len(emotion) != EXPECTED_EMOTION_ROWS:
        raise ValueError(
            "Emotion manifest row count does not match the current "
            "authoritative cleaned manifest. "
            f"Expected {EXPECTED_EMOTION_ROWS:,}, found {len(emotion):,}. "
            "Do not split a stale or partially rebuilt manifest."
        )

    if len(sarcasm) != EXPECTED_SARCASM_ROWS:
        raise ValueError(
            "Sarcasm manifest row count does not match the current "
            "authoritative cleaned manifest. "
            f"Expected {EXPECTED_SARCASM_ROWS:,}, found {len(sarcasm):,}. "
            "Do not split a stale or partially rebuilt manifest."
        )

    # Required active emotion datasets must all be represented.
    present_emotion_datasets = set(
        emotion["dataset"].dropna().astype(str).str.strip().unique()
    )
    missing_required = sorted(
        REQUIRED_EMOTION_DATASETS - present_emotion_datasets
    )
    if missing_required:
        raise ValueError(
            "Required emotion datasets are absent from the manifest: "
            f"{missing_required}"
        )

    # GoEmotions must remain present as text data.
    ge = emotion[
        emotion["dataset"].astype(str).str.strip().eq("GoEmotions")
    ].copy()

    if ge.empty:
        raise ValueError(
            "GoEmotions is absent from the emotion manifest."
        )

    if len(ge) != EXPECTED_GOEMOTIONS_FINAL_ROWS:
        raise ValueError(
            "GoEmotions final-manifest row count is not the current "
            "authoritative value. "
            f"Expected {EXPECTED_GOEMOTIONS_FINAL_ROWS:,}, "
            f"found {len(ge):,}. "
            "Do not split a stale or partially rebuilt emotion manifest."
        )

    if "modality" not in ge.columns:
        raise ValueError(
            "GoEmotions rows are missing the required modality column."
        )

    ge_modalities = set(
        ge["modality"].dropna().astype(str).str.strip().str.lower()
    )

    if ge_modalities != {"text"}:
        raise ValueError(
            "GoEmotions modality integrity failed. "
            f"Expected only text, found {sorted(ge_modalities)}."
        )

    # The corrected cleaner preserves the actual text payload.
    if "text" not in ge.columns:
        raise ValueError(
            "GoEmotions text payload column is missing from the final "
            "emotion manifest."
        )

    if ge["text"].fillna("").astype(str).str.strip().eq("").any():
        raise ValueError(
            "GoEmotions contains empty text payloads in the final manifest."
        )

    print(
        f"GoEmotions final rows: {len(ge):,} | "
        "modality=text | text payload: PASS"
    )

    # ------------------------------------------------------------
    # Architectural separation
    # ------------------------------------------------------------

    assert (
        "emotion" not in sarcasm.columns
    ), (
        "Sarcasm manifest unexpectedly contains "
        "an emotion target."
    )

    assert (
        "mapped_emotion" not in sarcasm.columns
    ), (
        "Sarcasm manifest unexpectedly contains "
        "a mapped_emotion target."
    )

    return emotion, sarcasm


# ================================================================
# Execute
# ================================================================

emotion_df, sarcasm_df = load_manifests()

C:\Users\Acer\AppData\Local\Temp\ipykernel_10200\688573017.py:46: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Emotion manifest rows: 146,335
Sarcasm manifest rows: 29,183
Emotion label column normalized from: mapped_emotion
Sarcasm label column normalized from: sarcasm
Emotion unique split groups: 127,186
Sarcasm unique split groups: 26,744
GoEmotions final rows: 50,831 | modality=text | text payload: PASS


In [4]:
# EDA for the current authoritative emotion and sarcasm manifests
def perform_eda(emotion, sarcasm):
    print("Generating EDA reports...")

    plt.figure(figsize=(12, 6))
    sns.countplot(data=emotion, x="emotion", hue="modality", order=EMOTION_CLASSES)
    plt.title("Emotion Distribution across Modalities")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(EDA_DIR / "emotion_class_distribution_by_modality.png")
    plt.close()

    plt.figure(figsize=(8, 5))
    sns.countplot(data=sarcasm, x="sarcasm_label", order=SARCASM_CLASSES)
    plt.title("Sarcasm Distribution")
    plt.tight_layout()
    plt.savefig(EDA_DIR / "sarcasm_class_distribution.png")
    plt.close()

    emotion.isnull().sum().to_csv(EDA_DIR / "emotion_missing_values.csv")
    sarcasm.isnull().sum().to_csv(EDA_DIR / "sarcasm_missing_values.csv")

    print(f"Emotion classes: {emotion['emotion'].value_counts().to_dict()}")
    print(f"Sarcasm classes: {sarcasm['sarcasm_label'].value_counts().to_dict()}")
    print(f"EDA saved to: {EDA_DIR}")

perform_eda(emotion_df, sarcasm_df)


Generating EDA reports...
Emotion classes: {'neutral': 40251, 'happiness': 34249, 'anger': 21411, 'sadness': 19523, 'surprise': 11899, 'fear': 11764, 'disgust': 7238}
Sarcasm classes: {'non_sarcastic': 15288, 'sarcastic': 13895}
EDA saved to: C:\New folder\New Emodect\reports\eda


In [5]:
# ================================================================
# Stratified + group-aware 70/15/15 split with exact final sizes
# ================================================================

def _repair_split_sizes(
    work,
    target_col,
    target_sizes,
):
    """
    Repair the deterministic SGKF allocation to the project's
    authoritative exact row counts.

    Only singleton split groups may be moved during repair.
    Therefore no reliable multi-row subject/session group is
    broken across splits.

    The repair is deterministic:
      - source/target selection is fixed by row counts
      - candidate rows are ordered by sample_id
      - class-wise quotas are deterministic

    This is a size-repair step only; it does not rewrite the
    upstream manifest.
    """
    current = work["split"].value_counts().reindex(
        ["train", "validation", "test"],
        fill_value=0,
    ).astype(int)

    if current.to_dict() == target_sizes:
        return work

    # Identify groups that contain exactly one row.
    group_sizes = work.groupby("split_group", dropna=False).size()
    singleton_groups = set(
        group_sizes[group_sizes.eq(1)].index
    )

    # Work on a deterministic copy.
    repaired = work.copy()

    # We only need to repair the current configuration where
    # validation is under target and train/test are over target.
    deficits = {
        split: int(target_sizes[split] - current[split])
        for split in ["train", "validation", "test"]
    }

    if sum(deficits.values()) != 0:
        raise RuntimeError("Target split sizes do not conserve total rows.")

    # General deterministic repair loop. At each step, move singleton
    # groups from an overfull split to an underfull split. When several
    # classes are possible, choose candidates using the target-class
    # proportions for the receiving split, then sample_id order.
    classes = list(
        repaired[target_col].dropna().astype(str).unique()
    )
    classes = sorted(classes)

    for destination in ["train", "validation", "test"]:
        need = deficits[destination]
        if need <= 0:
            continue

        sources = [
            s for s in ["train", "validation", "test"]
            if deficits[s] < 0
        ]

        # Desired class proportions in the destination based on the
        # complete manifest.
        overall_counts = (
            repaired[target_col]
            .astype(str)
            .value_counts()
            .reindex(classes, fill_value=0)
        )
        overall_props = overall_counts / overall_counts.sum()

        # Allocate the destination's missing rows across classes.
        raw = overall_props * need
        class_quota = np.floor(raw.to_numpy()).astype(int)
        class_quota = pd.Series(class_quota, index=raw.index, dtype=int)
        remainder = int(need - class_quota.sum())
        if remainder:
            fractional = (raw - class_quota).sort_values(
                ascending=False,
                kind="mergesort",
            )
            for cls in fractional.index[:remainder]:
                class_quota.loc[cls] += 1

        remaining_need = need

        for cls in classes:
            quota = int(class_quota.loc[cls])
            if quota <= 0:
                continue

            # Prefer the first available overfull source in a fixed order.
            selected_total = 0
            for source in sources:
                if deficits[source] >= 0:
                    continue

                available = repaired[
                    repaired["split"].eq(source)
                    & repaired["split_group"].isin(singleton_groups)
                    & repaired[target_col].astype(str).eq(cls)
                ].sort_values(
                    ["sample_id"] if "sample_id" in repaired.columns
                    else [target_col]
                )

                take = min(
                    quota - selected_total,
                    len(available),
                    -deficits[source],
                )

                if take <= 0:
                    continue

                selected_idx = available.index[:take]
                repaired.loc[selected_idx, "split"] = destination
                deficits[source] += int(take)
                deficits[destination] -= int(take)
                selected_total += int(take)

                if selected_total >= quota:
                    break

            if selected_total < quota:
                raise RuntimeError(
                    f"Unable to repair {destination} split to the exact "
                    f"target size without breaking a split group. "
                    f"Needed {quota}, selected {selected_total} for class {cls}."
                )

            remaining_need -= selected_total

        if remaining_need != 0:
            raise RuntimeError(
                f"Unable to satisfy exact {destination} size target; "
                f"{remaining_need} rows remain."
            )

    final_counts = (
        repaired["split"]
        .value_counts()
        .reindex(["train", "validation", "test"], fill_value=0)
        .astype(int)
        .to_dict()
    )

    if final_counts != target_sizes:
        raise RuntimeError(
            f"Exact split-size repair failed. "
            f"Expected {target_sizes}, found {final_counts}."
        )

    return repaired


def stratified_group_split(
    df,
    target_col,
    output_prefix,
    n_splits=20,
    random_state=42,
    target_sizes=None,
):
    """
    Create deterministic stratified/group-aware splits.

    Base allocation:
        StratifiedGroupKFold(
            n_splits=20,
            shuffle=True,
            random_state=42
        )

    Initial fold allocation:
        folds 0-13  -> train
        folds 14-16 -> validation
        folds 17-19 -> test

    Final-size repair:
        If target_sizes are supplied and the base SGKF allocation does
        not exactly match them, only singleton split groups are moved.
        Multi-row groups remain intact.

    This guarantees exact downstream row-count contracts without
    splitting a reliable subject/session group.
    """

    required = {target_col, "split_group"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Required columns missing for {output_prefix} split: {sorted(missing)}"
        )

    work = df.copy().reset_index(drop=True)

    if work.empty:
        raise ValueError(f"{output_prefix} manifest is empty.")

    if work[target_col].isna().any():
        raise ValueError(f"Missing target labels found in {output_prefix} data.")

    if work["split_group"].isna().any():
        raise ValueError(f"Missing split groups found in {output_prefix} data.")

    if work["split_group"].astype(str).str.strip().eq("").any():
        raise ValueError(f"Blank split groups found in {output_prefix} data.")

    if not work[target_col].isin(
        EMOTION_CLASSES if target_col == "emotion" else SARCASM_CLASSES
    ).all():
        raise ValueError(f"Unexpected labels found in {output_prefix} data.")

    if n_splits != 20:
        raise ValueError(
            "This project split configuration is fixed at n_splits=20."
        )

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    fold_id = np.full(len(work), -1, dtype=int)

    for fold, (_, test_idx) in enumerate(
        sgkf.split(
            work,
            work[target_col],
            groups=work["split_group"],
        )
    ):
        fold_id[test_idx] = fold

    if np.any(fold_id < 0):
        raise RuntimeError("Some rows were not assigned to a split fold.")

    work["_split_fold"] = fold_id

    work["split"] = np.select(
        [
            work["_split_fold"].between(0, 13),
            work["_split_fold"].between(14, 16),
            work["_split_fold"].between(17, 19),
        ],
        ["train", "validation", "test"],
        default="UNASSIGNED",
    )

    if (work["split"] == "UNASSIGNED").any():
        raise RuntimeError("UNASSIGNED rows detected.")

    base_counts = (
        work["split"]
        .value_counts()
        .reindex(["train", "validation", "test"], fill_value=0)
        .astype(int)
        .to_dict()
    )

    if target_sizes is not None:
        target_sizes = {
            key: int(target_sizes[key])
            for key in ["train", "validation", "test"]
        }

        if sum(target_sizes.values()) != len(work):
            raise ValueError(
                f"{output_prefix}: target split sizes do not sum to "
                f"manifest size. Targets={target_sizes}, rows={len(work):,}."
            )

        work = _repair_split_sizes(
            work,
            target_col,
            target_sizes,
        )

    train = work.loc[work["split"].eq("train")].copy()
    val = work.loc[work["split"].eq("validation")].copy()
    test = work.loc[work["split"].eq("test")].copy()

    # ------------------------------------------------------------
    # Group leakage guard
    # ------------------------------------------------------------

    train_groups = set(train["split_group"])
    val_groups = set(val["split_group"])
    test_groups = set(test["split_group"])

    if not train_groups.isdisjoint(val_groups):
        raise RuntimeError(f"{output_prefix}: train/validation group leakage.")
    if not train_groups.isdisjoint(test_groups):
        raise RuntimeError(f"{output_prefix}: train/test group leakage.")
    if not val_groups.isdisjoint(test_groups):
        raise RuntimeError(f"{output_prefix}: validation/test group leakage.")

    # ------------------------------------------------------------
    # Row conservation guard
    # ------------------------------------------------------------

    if len(train) + len(val) + len(test) != len(work):
        raise RuntimeError(f"{output_prefix}: split row conservation failed.")

    if target_sizes is not None:
        actual_sizes = {
            "train": len(train),
            "validation": len(val),
            "test": len(test),
        }
        if actual_sizes != target_sizes:
            raise RuntimeError(
                f"{output_prefix}: exact target sizes failed. "
                f"Expected {target_sizes}, found {actual_sizes}."
            )

    # ------------------------------------------------------------
    # Save split-specific files and combined split manifest.
    # Do not expose internal split_group/fold columns downstream.
    # ------------------------------------------------------------

    keep = [
        c for c in work.columns
        if c not in {"split_group", "_split_fold"}
    ]

    train[keep].to_csv(
        SPLIT_DIR / f"{output_prefix}_train.csv",
        index=False,
    )
    val[keep].to_csv(
        SPLIT_DIR / f"{output_prefix}_validation.csv",
        index=False,
    )
    test[keep].to_csv(
        SPLIT_DIR / f"{output_prefix}_test.csv",
        index=False,
    )
    work[keep].to_csv(
        SPLIT_DIR / f"{output_prefix}_all_splits.csv",
        index=False,
    )

    persisted_total = sum(
        len(pd.read_csv(SPLIT_DIR / f"{output_prefix}_{suffix}.csv"))
        for suffix in ["train", "validation", "test"]
    )
    if persisted_total != len(work):
        raise RuntimeError(
            f"{output_prefix}: persisted split row conservation failed."
        )

    print(f"\n{output_prefix.upper()} SPLIT")
    print(f"Base SGKF allocation: {base_counts}")
    print(f"Final total:      {len(work):,}")
    print(f"Final train:      {len(train):,} ({len(train)/len(work):.2%})")
    print(f"Final validation: {len(val):,} ({len(val)/len(work):.2%})")
    print(f"Final test:       {len(test):,} ({len(test)/len(work):.2%})")
    print(
        f"Groups: train={len(train_groups):,}, "
        f"val={len(val_groups):,}, test={len(test_groups):,}"
    )

    return train, val, test


emotion_train, emotion_val, emotion_test = stratified_group_split(
    emotion_df,
    "emotion",
    "emotion",
    n_splits=20,
    random_state=42,
    target_sizes=EXPECTED_EMOTION_SPLITS,
)

sarcasm_train, sarcasm_val, sarcasm_test = stratified_group_split(
    sarcasm_df,
    "sarcasm_label",
    "sarcasm",
    n_splits=20,
    random_state=42,
    target_sizes=EXPECTED_SARCASM_SPLITS,
)


C:\Users\Acer\AppData\Local\Temp\ipykernel_10200\1969617638.py:357: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  len(pd.read_csv(SPLIT_DIR / f"{output_prefix}_{suffix}.csv"))



EMOTION SPLIT
Base SGKF allocation: {'train': 100286, 'validation': 23339, 'test': 22710}
Final total:      146,335
Final train:      99,961 (68.31%)
Final validation: 24,679 (16.86%)
Final test:       21,695 (14.83%)
Groups: train=86,641, val=21,509, test=19,036

SARCASM SPLIT
Base SGKF allocation: {'train': 20254, 'validation': 4479, 'test': 4450}
Final total:      29,183
Final train:      20,254 (69.40%)
Final validation: 4,479 (15.35%)
Final test:       4,450 (15.25%)
Groups: train=18,572, val=4,090, test=4,082


In [6]:
# ================================================================
# Final QA: class balance, group leakage, target separation,
# row conservation, and modality accounting
# ================================================================

def split_qa(
    original,
    train,
    val,
    test,
    target_col,
    expected_labels,
    name,
):
    parts = {
        "train": train,
        "validation": val,
        "test": test,
    }

    # Row conservation against the original manifest.
    split_total = len(train) + len(val) + len(test)
    assert split_total == len(original), (
        f"{name}: split rows {split_total:,} != original rows {len(original):,}"
    )

    # All expected classes must appear in every split.
    for split_name, frame in parts.items():
        observed = set(frame[target_col].dropna().unique())
        missing = set(expected_labels) - observed
        assert not missing, (
            f"{name}: {split_name} is missing classes: {sorted(missing)}"
        )

    # Group leakage.
    group_sets = {
        key: set(frame["split_group"])
        for key, frame in parts.items()
    }

    assert group_sets["train"].isdisjoint(group_sets["validation"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["validation"].isdisjoint(group_sets["test"])

    print(f"\n{name} QA")
    print("-" * 60)
    print("Class counts:")
    counts = pd.DataFrame({
        split_name: frame[target_col].value_counts().reindex(
            expected_labels, fill_value=0
        )
        for split_name, frame in parts.items()
    })
    print(counts.to_string())

    print("\nClass percentages:")
    percentages = counts.div(counts.sum(axis=0), axis=1) * 100
    print(percentages.round(2).to_string())

    print(
        f"\nRows: total={len(original):,}, "
        f"train={len(train):,}, validation={len(val):,}, test={len(test):,}"
    )
    print(
        f"Groups: train={len(group_sets['train']):,}, "
        f"validation={len(group_sets['validation']):,}, "
        f"test={len(group_sets['test']):,}"
    )
    print("Group leakage: PASS")
    print("All expected classes present in every split: PASS")


split_qa(
    emotion_df,
    emotion_train,
    emotion_val,
    emotion_test,
    "emotion",
    EMOTION_CLASSES,
    "Emotion",
)

split_qa(
    sarcasm_df,
    sarcasm_train,
    sarcasm_val,
    sarcasm_test,
    "sarcasm_label",
    SARCASM_CLASSES,
    "Sarcasm",
)

# Explicit architectural guard:
# sarcasm never enters the seven-class emotion target.
assert "sarcasm_label" not in emotion_train.columns
assert "emotion" not in sarcasm_train.columns
print("\nTarget separation: PASS")

# GoEmotions must remain in the emotion pipeline as text.
goemotions_total = emotion_df[
    emotion_df["dataset"].astype(str).str.strip().eq("GoEmotions")
]
assert len(goemotions_total) > 0
if "modality" in goemotions_total.columns:
    assert (
        goemotions_total["modality"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("text")
        .all()
    ), "GoEmotions modality integrity failed."

print(f"GoEmotions text modality: PASS ({len(goemotions_total):,} rows)")

# The cleaned manifest intentionally contains no SAMM samples.
if "dataset" in emotion_df.columns:
    assert not emotion_df["dataset"].astype(str).str.strip().eq("SAMM").any()
print("SAMM excluded from active emotion manifest: PASS")

# Important scope note:
# datasets without defensible subject identity use sample-only groups.
# Therefore this split guarantees group isolation where reliable group
# metadata exists, but does not claim subject-disjoint evaluation for
# datasets whose source lacks subject identity metadata.
print(
    "Subject-disjointness scope: guaranteed where defensible group metadata exists; "
    "sample-only datasets are not claimed as subject-disjoint."
)



Emotion QA
------------------------------------------------------------
Class counts:
           train  validation  test
emotion                           
anger      14381        3582  3448
disgust     4740        1423  1075
fear        7753        2225  1786
happiness  23723        5474  5052
sadness    13472        3260  2791
surprise    8002        2156  1741
neutral    27890        6559  5802

Class percentages:
           train  validation   test
emotion                            
anger      14.39       14.51  15.89
disgust     4.74        5.77   4.96
fear        7.76        9.02   8.23
happiness  23.73       22.18  23.29
sadness    13.48       13.21  12.86
surprise    8.01        8.74   8.02
neutral    27.90       26.58  26.74

Rows: total=146,335, train=99,961, validation=24,679, test=21,695
Groups: train=86,641, validation=21,509, test=19,036
Group leakage: PASS
All expected classes present in every split: PASS

Sarcasm QA
----------------------------------------------------

In [7]:
# ================================================================
# Final summary and paths
# ================================================================

summary = pd.DataFrame([
    {
        "pipeline": "emotion",
        "rows": len(emotion_df),
        "train": len(emotion_train),
        "validation": len(emotion_val),
        "test": len(emotion_test),
        "target": "7-class emotion",
    },
    {
        "pipeline": "sarcasm",
        "rows": len(sarcasm_df),
        "train": len(sarcasm_train),
        "validation": len(sarcasm_val),
        "test": len(sarcasm_test),
        "target": "sarcastic / non_sarcastic",
    },
])
display(summary)

print("\nAuthoritative manifest sizes:")
print(f"Emotion : {len(emotion_df):,} (expected {EXPECTED_EMOTION_ROWS:,})")
print(f"Sarcasm : {len(sarcasm_df):,} (expected {EXPECTED_SARCASM_ROWS:,})")

print("\nFinal split targets:")
print(f"Emotion: {EXPECTED_EMOTION_SPLITS}")
print(f"Sarcasm: {EXPECTED_SARCASM_SPLITS}")

print("\nSplit configuration:")
print("StratifiedGroupKFold: n_splits=20, shuffle=True, random_state=42")
print("Fold allocation: 0-13 train, 14-16 validation, 17-19 test")

print("\nEmotion split files:")
print(SPLIT_DIR / "emotion_train.csv")
print(SPLIT_DIR / "emotion_validation.csv")
print(SPLIT_DIR / "emotion_test.csv")
print(SPLIT_DIR / "emotion_all_splits.csv")

print("\nSarcasm split files:")
print(SPLIT_DIR / "sarcasm_train.csv")
print(SPLIT_DIR / "sarcasm_validation.csv")
print(SPLIT_DIR / "sarcasm_test.csv")
print(SPLIT_DIR / "sarcasm_all_splits.csv")

# Final authoritative manifest-size gate.
assert len(emotion_df) == EXPECTED_EMOTION_ROWS
assert len(sarcasm_df) == EXPECTED_SARCASM_ROWS

final_goemotions_rows = int(
    emotion_df["dataset"]
    .astype(str)
    .str.strip()
    .eq("GoEmotions")
    .sum()
)
assert final_goemotions_rows == EXPECTED_GOEMOTIONS_FINAL_ROWS

print(
    f"GoEmotions final manifest rows: "
    f"{final_goemotions_rows:,} (expected {EXPECTED_GOEMOTIONS_FINAL_ROWS:,})"
)
print("Authoritative manifest size gate: PASS")

print("\nFINAL SPLIT GATE: PASS")
print("Quadra target architecture: 7-class emotion + independent event-driven sarcasm detector.")


,pipeline,rows,train,validation,test,target
0,emotion,146335,99961,24679,21695,7-class emotion
1,sarcasm,29183,20254,4479,4450,sarcastic / non_sarcastic



Authoritative manifest sizes:
Emotion : 146,335 (expected 146,335)
Sarcasm : 29,183 (expected 29,183)

Final split targets:
Emotion: {'train': 99961, 'validation': 24679, 'test': 21695}
Sarcasm: {'train': 20254, 'validation': 4479, 'test': 4450}

Split configuration:
StratifiedGroupKFold: n_splits=20, shuffle=True, random_state=42
Fold allocation: 0-13 train, 14-16 validation, 17-19 test

Emotion split files:
C:\New folder\New Emodect\cleaned_metadata\splits\emotion_train.csv
C:\New folder\New Emodect\cleaned_metadata\splits\emotion_validation.csv
C:\New folder\New Emodect\cleaned_metadata\splits\emotion_test.csv
C:\New folder\New Emodect\cleaned_metadata\splits\emotion_all_splits.csv

Sarcasm split files:
C:\New folder\New Emodect\cleaned_metadata\splits\sarcasm_train.csv
C:\New folder\New Emodect\cleaned_metadata\splits\sarcasm_validation.csv
C:\New folder\New Emodect\cleaned_metadata\splits\sarcasm_test.csv
C:\New folder\New Emodect\cleaned_metadata\splits\sarcasm_all_splits.csv
Go